# PIBIC · Treinamento da MLP para suspensão de 1 GDL

Este estudo amplia o treino harmônico com degrau, impulso e força de ruído gaussiano filtrado em quatro bandas. O CSV expandido com 2.520 simulações acompanha o notebook. Execute todas as células para reproduzir as métricas e os gráficos.

A rede usa a arquitetura do treino original (128–128–64, SiLU, PyTorch), com novos pesos e quatro alvos em escala física: pico absoluto de deslocamento e RMS de deslocamento, velocidade e aceleração. As respostas são indicadores das simulações com força aplicada à massa em uma janela de 64 s, amostrada a 512 Hz, e amortecimento de 2%. Para o impulso ideal, a aceleração RMS se refere à resposta regular após o salto de velocidade.

Para cada combinação de sistema, banda e intensidade de ruído, calculamos a média aritmética de cada indicador entre três realizações. O desvio padrão amostral mede a variação observada entre essas realizações. Cada condição determinística entra uma vez. A avaliação usa famílias completas de parâmetros como grupos e compara a MLP com a referência das simulações.


In [ ]:
# @title Bibliotecas e reprodutibilidade
import base64
import gzip
import hashlib
import io
import json
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from torch import nn

SEMENTE = 42
torch.set_num_threads(2)
torch.use_deterministic_algorithms(True)
PASTA = Path('resultados_mlp_1gdl')
PASTA.mkdir(exist_ok=True)
ALVOS = ['x_pico_abs_m', 'x_rms_m', 'v_rms_m_s', 'a_rms_m_s2']
ROTULOS = dict(zip(ALVOS, ['Pico absoluto de x', 'RMS de x', 'RMS de v', 'RMS de a']))
CATEGORIAS = {
    'degrau': 'Degrau', 'impulso': 'Impulso', 'subressonante': 'Subressonante',
    'ressonante': 'Ressonante', 'suprarressonante': 'Suprarressonante', 'larga': 'Banda larga',
}


In [ ]:
# @title Carregar o CSV expandido de 2.520 simulações
CSV_EMBUTIDO = (
    '...'
)
dados_csv = gzip.decompress(base64.b64decode(CSV_EMBUTIDO))
assert hashlib.sha256(dados_csv).hexdigest() == '43b95bdb2feea3de9835051884980797f3289db23cea4bb68470b33500e65798'
banco = pd.read_csv(io.BytesIO(dados_csv))
print(f"{len(banco):,} simulações carregadas | {banco.sistema_id.nunique()} sistemas | "
      f"{banco.grupo_origem.nunique()} famílias")


In [ ]:
# @title Calcular a média e a dispersão por condição
assert len(banco) == 2520 and banco.id_simulacao.is_unique
assert banco.grupo_origem.nunique() == 10 and banco.sistema_id.nunique() == 60
assert np.isclose(banco.zeta, .02).all() and np.isclose(banco.tf_s, 64).all()
assert np.isclose(banco.fs_Hz, 512).all() and banco[['x0_m', 'v0_m_s']].eq(0).all().all()
assert np.isfinite(banco[ALVOS]).all().all() and banco[ALVOS].gt(0).all().all()

base = banco.assign(banda=banco.banda.fillna(''))
chaves = ['grupo_origem', 'sistema_id', 'escala_ref_m', 'excitacao', 'banda']
entradas = ['m_kg', 'k_N_m', 'zeta', 'tf_s', 'F_degrau_N', 'J_impulso_N_s',
            'F_ruido_rms_N', 'fmin_Hz', 'fmax_Hz']
grupos = base.groupby(chaves, sort=True)
assert grupos[entradas].nunique(dropna=False).le(1).all().all()
agregacoes = {coluna: (coluna, 'first') for coluna in entradas}
agregacoes.update({alvo: (alvo, 'mean') for alvo in ALVOS})
agregacoes.update({f'dp_{alvo}': (alvo, 'std') for alvo in ALVOS})
dados = grupos.agg(**agregacoes, n_realizacoes=('id_simulacao', 'size'),
                   sementes_distintas=('semente_ruido', 'nunique')).reset_index()
ruido = dados.excitacao.eq('ruido')
assert dados.loc[ruido, 'n_realizacoes'].eq(3).all()
assert dados.loc[ruido, 'sementes_distintas'].eq(3).all()
assert dados.loc[~ruido, 'n_realizacoes'].eq(1).all()
assert base.loc[base.excitacao.eq('ruido')].groupby('semente_ruido').grupo_origem.nunique().eq(1).all()
dados['categoria'] = dados.banda.where(ruido, dados.excitacao)
assert len(dados) == 1080 and dados.categoria.value_counts().eq(180).all()
assert set(dados.categoria) == set(CATEGORIAS)
display(dados.groupby('categoria').n_realizacoes.agg(condicoes='size', simulacoes='sum').reindex(CATEGORIAS))


## Entradas físicas e avaliação por família

Massa e rigidez dão a frequência natural angular $\omega_n=\sqrt{k/m}$. A escala de deslocamento $S$ vale $F/k$ para o degrau e para o ruído (força RMS), e $J/(m\omega_n)$ para o impulso. Como as equações são lineares e as condições iniciais são nulas, treinamos a rede com os alvos divididos por $S$, $S$, $S\omega_n$ e $S\omega_n^2$, respectivamente. Multiplicamos pelas mesmas escalas para avaliar os resultados no SI.

A rede recebe o tipo de excitação, a duração adimensional $T\omega_n$ em log e os limites da banda em Hz divididos pela frequência natural em Hz. Para degrau e impulso, os limites da banda valem zero. As três intensidades mantêm o mesmo peso no treino e nas métricas, mas não são variações independentes do comportamento de um sistema linear.

Usamos cinco rodadas. Cada rodada reserva duas famílias para teste, duas para validação e seis para treino. Cada família aparece no teste uma única vez. A normalização é ajustada com o treino de cada rodada e a melhor época é escolhida pela perda de validação. A mediana dos alvos adimensionais por categoria, calculada no treino, serve de referência simples. Erros sobre as médias de ruído medem desempenho frente às três realizações do banco, que também têm incerteza amostral.


In [ ]:
# @title Preparar as entradas e a divisão por grupo de origem
def preparar_entradas(condicoes):
    massa = condicoes.m_kg.to_numpy()
    rigidez = condicoes.k_N_m.to_numpy()
    wn = np.sqrt(rigidez / massa)
    fn = wn / (2 * np.pi)
    escala = (
        condicoes.F_degrau_N.fillna(0).to_numpy() / rigidez
        + condicoes.J_impulso_N_s.fillna(0).to_numpy() / (massa * wn)
        + condicoes.F_ruido_rms_N.fillna(0).to_numpy() / rigidez
    )
    escalas_y = escala[:, None] * np.column_stack([np.ones_like(wn), np.ones_like(wn), wn, wn**2])
    atributos = pd.DataFrame({
        'log10_T_wn': np.log10(condicoes.tf_s * wn),
        'fmin_sobre_fn': np.round(condicoes.fmin_Hz.fillna(0) / fn, 10),
        'fmax_sobre_fn': np.round(condicoes.fmax_Hz.fillna(0) / fn, 10),
        **{tipo: condicoes.excitacao.eq(tipo).astype(float) for tipo in ['degrau', 'impulso', 'ruido']},
    })
    assert np.isfinite(atributos).all().all() and np.isfinite(escalas_y).all() and (escala > 0).all()
    return atributos, escalas_y

atributos, escalas_y = preparar_entradas(dados)
x = atributos.to_numpy()
y_real = dados[ALVOS].to_numpy()
y_log = np.log10(y_real / escalas_y)
familias = dados.grupo_origem.to_numpy()
testes = [teste for _, teste in GroupKFold(n_splits=5).split(x, groups=familias)]
particoes = []
for rodada, teste in enumerate(testes):
    validacao = testes[(rodada + 1) % len(testes)]
    treino = np.setdiff1d(np.arange(len(dados)), np.r_[validacao, teste])
    grupos_tr, grupos_va, grupos_te = [set(familias[idx]) for idx in (treino, validacao, teste)]
    assert (len(grupos_tr), len(grupos_va), len(grupos_te)) == (6, 2, 2)
    assert not (grupos_tr & grupos_va or grupos_tr & grupos_te or grupos_va & grupos_te)
    particoes.append(dict(rodada=rodada + 1, treino=' '.join(sorted(grupos_tr)),
                         validacao=' '.join(sorted(grupos_va)), teste=' '.join(sorted(grupos_te))))
particoes = pd.DataFrame(particoes)
display(particoes)


In [ ]:
# @title Treinar cinco MLPs e gerar previsões fora do treino
class SuspensionMLP(nn.Module):
    def __init__(self, n_entradas, n_alvos):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_entradas, 128), nn.SiLU(),
            nn.Linear(128, 128), nn.SiLU(),
            nn.Linear(128, 64), nn.SiLU(), nn.Linear(64, n_alvos),
        )

    def forward(self, valores):
        return self.net(valores)


def tensor(valores):
    return torch.as_tensor(valores, dtype=torch.float32)


def treinar_rodada(treino, validacao, teste, rodada):
    torch.manual_seed(SEMENTE + rodada)
    scaler_x = StandardScaler().fit(x[treino])
    scaler_y = StandardScaler().fit(y_log[treino])
    x_tr, x_va, x_te = (tensor(scaler_x.transform(x[idx])) for idx in (treino, validacao, teste))
    y_tr, y_va = (tensor(scaler_y.transform(y_log[idx])) for idx in (treino, validacao))
    modelo = SuspensionMLP(x.shape[1], len(ALVOS))
    perda = nn.MSELoss()
    otimizador = torch.optim.AdamW(modelo.parameters(), lr=3e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(otimizador, factor=.5, patience=15)
    historico = []
    melhor_val, melhor_epoca = float('inf'), 0
    for epoca in range(1, 301):
        modelo.train()
        for lote in torch.randperm(len(x_tr)).split(64):
            otimizador.zero_grad(set_to_none=True)
            perda(modelo(x_tr[lote]), y_tr[lote]).backward()
            otimizador.step()
        modelo.eval()
        with torch.no_grad():
            mse_tr = perda(modelo(x_tr), y_tr).item()
            mse_va = perda(modelo(x_va), y_va).item()
        historico.append(dict(rodada=rodada, epoca=epoca, mse_treino=mse_tr, mse_validacao=mse_va))
        if mse_va < melhor_val:
            melhor_val, melhor_epoca = mse_va, epoca
            melhor_estado = {nome: valor.detach().clone() for nome, valor in modelo.state_dict().items()}
        scheduler.step(mse_va)
        if epoca - melhor_epoca >= 45:
            break

    modelo.load_state_dict(melhor_estado)
    with torch.no_grad():
        pred_log = scaler_y.inverse_transform(modelo(x_te).numpy())
    torch.save({
        'pesos': melhor_estado, 'atributos': list(atributos.columns), 'alvos': ALVOS,
        'media_x': scaler_x.mean_.tolist(), 'escala_x': scaler_x.scale_.tolist(),
        'media_y_log': scaler_y.mean_.tolist(), 'escala_y_log': scaler_y.scale_.tolist(),
        'familias_treino': sorted(set(familias[treino])),
        'familias_validacao': sorted(set(familias[validacao])),
        'familias_teste': sorted(set(familias[teste])),
        'melhor_epoca': melhor_epoca,
    }, PASTA / f'mlp_rodada_{rodada}.pt')
    print(f'Rodada {rodada}/5 | melhor época {melhor_epoca} de {epoca} | MSE validação {melhor_val:.5f}')
    return np.power(10., pred_log) * escalas_y[teste], historico


y_pred = np.full_like(y_real, np.nan)
y_referencia = np.full_like(y_real, np.nan)
rodadas_teste = np.zeros(len(dados), dtype=int)
historicos = []
for rodada, teste in enumerate(testes, start=1):
    validacao = testes[rodada % len(testes)]
    treino = np.setdiff1d(np.arange(len(dados)), np.r_[validacao, teste])
    y_pred[teste], historico = treinar_rodada(treino, validacao, teste, rodada)
    historicos.extend(historico)
    medianas = pd.DataFrame(y_log[treino], index=dados.categoria.iloc[treino], columns=ALVOS).groupby(level=0).median()
    y_referencia[teste] = np.power(10., medianas.loc[dados.categoria.iloc[teste]].to_numpy()) * escalas_y[teste]
    assert not rodadas_teste[teste].any()
    rodadas_teste[teste] = rodada

assert np.isfinite(y_pred).all() and np.isfinite(y_referencia).all() and rodadas_teste.all()
historicos = pd.DataFrame(historicos)


In [ ]:
# @title Medir o erro de teste e a dispersão do ruído
registros = []
categorias_teste = ['geral', *CATEGORIAS]
for categoria in categorias_teste:
    selecao = np.ones(len(dados), dtype=bool) if categoria == 'geral' else dados.categoria.eq(categoria).to_numpy()
    for j, alvo in enumerate(ALVOS):
        real, pred, ref = y_real[selecao, j], y_pred[selecao, j], y_referencia[selecao, j]
        erro = np.abs(pred - real)
        erro_pct = 100 * erro / real
        desvio = dados.loc[selecao, f'dp_{alvo}'].to_numpy() / real * 100
        dispersao = np.nanmedian(desvio) if np.isfinite(desvio).any() and categoria != 'geral' else np.nan
        registros.append(dict(categoria=categoria, alvo=alvo, n=int(selecao.sum()),
            mae_SI=erro.mean(), mape_pct=erro_pct.mean(), p90_erro_pct=np.percentile(erro_pct, 90),
            r2=r2_score(real, pred), mape_referencia_pct=np.mean(100 * np.abs(ref - real) / real),
            mediana_cv_ruido_pct=dispersao))
metricas = pd.DataFrame(registros)
previsoes = dados.assign(rodada_teste=rodadas_teste).copy()
for j, alvo in enumerate(ALVOS):
    previsoes[f'prev_{alvo}'] = y_pred[:, j]
    previsoes[f'ref_mediana_{alvo}'] = y_referencia[:, j]
familia_resultados = []
for familia, tabela in previsoes.groupby('grupo_origem'):
    for alvo in ALVOS:
        mape = (100 * (tabela[f'prev_{alvo}'] / tabela[alvo] - 1).abs()).mean()
        familia_resultados.append({'familia': familia, 'alvo': alvo, 'mape_pct': mape})
familia_resultados = pd.DataFrame(familia_resultados)

def tabela_por_categoria(coluna):
    tabela = metricas.pivot(index='categoria', columns='alvo', values=coluna)
    return tabela.reindex(index=categorias_teste, columns=ALVOS).rename(
        index={**CATEGORIAS, 'geral': 'Geral'}, columns=ROTULOS)

print('MAPE (%) da MLP nas famílias reservadas')
display(tabela_por_categoria('mape_pct').round(2))
print('MAPE (%) da referência baseada nas medianas do treino')
display(tabela_por_categoria('mape_referencia_pct').round(2))
print('Dispersão do ruído: mediana do desvio padrão / média das três realizações (%)')
display(tabela_por_categoria('mediana_cv_ruido_pct').dropna(how='all').round(2))


In [ ]:
# @title Curvas de aprendizagem, paridade e erros por condição
cores = ['#2166ac', '#d6604d', '#4393c3', '#762a83', '#228450', '#b66a0c']
fig, eixos = plt.subplots(2, 3, figsize=(12, 6.8), layout='constrained', sharey=True)
for ax, (rodada, hist) in zip(eixos.flat, historicos.groupby('rodada')):
    ax.semilogy(hist.epoca, hist.mse_treino, color='#2166ac', label='Treino')
    ax.semilogy(hist.epoca, hist.mse_validacao, color='#d6604d', label='Validação')
    melhor = hist.loc[hist.mse_validacao.idxmin(), 'epoca']
    ax.axvline(melhor, color='0.35', linestyle=':', linewidth=1, label='Melhor época')
    ax.set(title=f'Rodada {rodada}', xlabel='Época', ylabel='MSE em escala de treino')
    ax.grid(alpha=.2)
eixos.flat[-1].axis('off')
eixos.flat[0].legend(fontsize=8)
fig.savefig(PASTA / 'curvas_aprendizagem.png', dpi=150)
plt.show()

fig, eixos = plt.subplots(2, 2, figsize=(10.8, 8.2), layout='constrained')
for j, (ax, alvo, unidade, fator) in enumerate(zip(eixos.flat, ALVOS, ['mm', 'mm', 'm/s', 'm/s²'], [1000, 1000, 1, 1])):
    sim, pred = fator * y_real[:, j], fator * y_pred[:, j]
    for (categoria, nome), cor in zip(CATEGORIAS.items(), cores):
        sel = dados.categoria.eq(categoria)
        ax.scatter(sim[sel], pred[sel], s=13, alpha=.6, color=cor, label=nome)
    lo, hi = min(sim.min(), pred.min()), max(sim.max(), pred.max())
    ax.plot([lo, hi], [lo, hi], '--', color='0.25', linewidth=1)
    ax.set(xscale='log', yscale='log', title=ROTULOS[alvo],
           xlabel=f'Referência simulada ({unidade})', ylabel=f'MLP ({unidade})')
    ax.set_aspect('equal', adjustable='box')
    ax.grid(alpha=.2, which='both')
eixos.flat[0].legend(fontsize=7, loc='lower right')
fig.savefig(PASTA / 'paridade_teste.png', dpi=150)
plt.show()

fig, eixos = plt.subplots(2, 2, figsize=(11, 6), layout='constrained')
for ax, alvo in zip(eixos.flat, ALVOS):
    linhas = metricas.loc[metricas.alvo.eq(alvo)].set_index('categoria').loc[list(CATEGORIAS)]
    pos = np.arange(len(CATEGORIAS))
    ax.bar(pos - .18, linhas.mape_pct, width=.36, color='#2166ac', label='MLP')
    ax.bar(pos + .18, linhas.mape_referencia_pct, width=.36, color='#888888', label='Mediana do treino')
    ax.set_xticks(pos, CATEGORIAS.values(), rotation=18, ha='right')
    ax.set(title=ROTULOS[alvo], ylabel='MAPE (%)')
    ax.grid(axis='y', alpha=.2)
eixos.flat[0].legend(fontsize=8)
fig.savefig(PASTA / 'erros_por_categoria.png', dpi=150)
plt.show()


In [ ]:
# @title Resumo do estudo e exportação dos resultados
gerais = metricas.loc[metricas.categoria.eq('geral')].set_index('alvo').loc[ALVOS]
detalhadas = metricas.loc[metricas.categoria.ne('geral')]
pior = detalhadas.loc[detalhadas.mape_pct.idxmax()]
dispersoes = metricas.dropna(subset=['mediana_cv_ruido_pct'])
dp_max = dispersoes.loc[dispersoes.mediana_cv_ruido_pct.idxmax()]
vence = int((detalhadas.mape_pct < detalhadas.mape_referencia_pct).sum())
relatorio = [
    '# PIBIC · Resultado da MLP de 1 GDL', '',
    'O estudo usou 2.520 simulações, 60 configurações e 10 famílias. Após a média das três realizações de ruído por condição, ficaram 1.080 condições, com 180 por categoria. Todas as condições tiveram uma previsão de teste feita por uma rede treinada sem a sua família.', '',
    '| Indicador | MAPE MLP | MAPE mediana | R² MLP |',
    '|---|---:|---:|---:|',
]
for alvo, linha in gerais.iterrows():
    relatorio.append(f'| {ROTULOS[alvo]} | {linha.mape_pct:.2f}% | {linha.mape_referencia_pct:.2f}% | {linha.r2:.4f} |')
relatorio.extend([
    '', f'A MLP teve MAPE menor que a mediana do treino em **{vence} de {len(detalhadas)}** combinações de categoria e indicador. O maior MAPE por categoria foi **{pior.mape_pct:.2f}%**, para **{ROTULOS[pior.alvo]}** em **{CATEGORIAS[pior.categoria]}**.',
    '', f'A maior dispersão relativa mediana entre as três realizações foi **{dp_max.mediana_cv_ruido_pct:.2f}%**, para **{ROTULOS[dp_max.alvo]}** em **{CATEGORIAS[dp_max.categoria]}**. Esse número é um coeficiente de variação amostral, calculado dentro de cada condição.',
    '', 'Os resultados valem para o oscilador linear simulado com amortecimento de 2%, condições iniciais nulas e quatro bandas fixadas em relação à frequência natural. A referência é a solução do simulador. A aplicação a medições físicas, outras taxas de amortecimento ou um modelo de 2 GDL exigirá nova avaliação.',
])
texto = '\n'.join(relatorio) + '\n'
(PASTA / 'relatorio_mlp_1gdl.md').write_text(texto, encoding='utf-8')
previsoes.to_csv(PASTA / 'previsoes_teste.csv', index=False)
metricas.to_csv(PASTA / 'metricas_por_categoria.csv', index=False)
familia_resultados.to_csv(PASTA / 'metricas_por_familia.csv', index=False)
particoes.to_csv(PASTA / 'particoes_por_familia.csv', index=False)
historicos.to_csv(PASTA / 'curvas_aprendizagem.csv', index=False)
(PASTA / 'proveniencia.json').write_text(json.dumps({
    'arquivo_fonte': 'banco_1gdl_expandido_2520.csv', 'sha256': hashlib.sha256(dados_csv).hexdigest(),
    'sementes_treino': [SEMENTE + rodada for rodada in range(1, 6)], 'torch': str(torch.__version__),
}, indent=2), encoding='utf-8')
arquivo_zip = shutil.make_archive('resultados_mlp_1gdl', 'zip', root_dir=PASTA)
display(Markdown(texto))
print('Treino e avaliação concluídos. Arquivo de resultados: resultados_mlp_1gdl.zip')


In [ ]:
# @title Baixar resultados (opcional)
baixar_zip = False # @param {type:"boolean"}
if baixar_zip:
    from google.colab import files
    files.download(arquivo_zip)
